# tt-mlir #9295 빌드 + lit test

**할 일은 딱 둘:**
1. 우상단 런타임(Runtime) → "런타임 유형 변경" → **CPU, 고RAM(High-RAM)** 선택 (GPU 불필요, LLVM 컴파일은 CPU-bound)
2. 메뉴 Runtime → **Run all** 클릭

이후로는 셀 다 자동 실행됨. 마지막 셀에서 lit test PASS/FAIL 결과 보여줌.


In [ ]:
!nproc
!free -h


In [ ]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq clang ninja-build cmake git python3.12-venv libgtest-dev libgmock-dev


In [ ]:
!rm -rf /content/tt-mlir
!git clone --branch fix-9295-multi-result-dram-fallback https://github.com/alexxony/tt-mlir.git /content/tt-mlir
%cd /content/tt-mlir
!git log --oneline -3


In [ ]:
import os
os.environ["TTMLIR_TOOLCHAIN_DIR"] = "/opt/ttmlir-toolchain/"
!mkdir -p /opt/ttmlir-toolchain
!chown -R $(whoami) /opt/ttmlir-toolchain


## Toolchain 빌드 (LLVM 등) — 가장 오래 걸림, 20~40분

In [ ]:
%cd /content/tt-mlir
!cmake -B env/build env -DCMAKE_C_COMPILER=clang -DCMAKE_CXX_COMPILER=clang++
!cmake --build env/build


## tt-mlir 본체 빌드 (런타임/opmodel 비활성화, lit test만 목적)

In [ ]:
%cd /content/tt-mlir
!bash -c "source env/activate && cmake -G Ninja -B build -DTTMLIR_ENABLE_RUNTIME=OFF -DTTMLIR_ENABLE_OPMODEL=OFF -DTTMLIR_ENABLE_BINDINGS_PYTHON=OFF -DCMAKE_BUILD_PARALLEL_LEVEL=$(nproc)"
!bash -c "source env/activate && cmake --build build -- -j$(nproc)"


## lit test 실행

In [ ]:
%cd /content/tt-mlir
!bash -c "source env/activate && cmake --build build -- check-ttmlir" 2>&1 | tee /content/lit_test_full.log | tail -100


In [ ]:
%cd /content/tt-mlir
!build/bin/llvm-lit test/ttmlir/Dialect/TTNN/optimizer/op_layout_fallbacks/multi_result_dram_fallback.mlir -v 2>&1 | tee /content/multi_result_test.log


## 결과 요약

In [ ]:
import re

print("=" * 60)
print("신규 테스트(multi_result_dram_fallback.mlir) 결과")
print("=" * 60)
with open("/content/multi_result_test.log") as f:
    print(f.read())

print("=" * 60)
print("전체 check-ttmlir 요약 (마지막 30줄)")
print("=" * 60)
with open("/content/lit_test_full.log") as f:
    lines = f.readlines()
    print("".join(lines[-30:]))


## 산출물 다운로드 (선택)

결과를 로컬(WSL)로 가져가고 싶으면 아래 셀 실행 후, `Files` 사이드바에서 `/content/backup_bundle` 우클릭 → 다운로드.
또는 그냥 위 결과 요약 셀 출력만 스크린샷/복사해서 알려줘도 됨.


In [ ]:
import shutil, os
os.makedirs("/content/backup_bundle", exist_ok=True)
for f in ["lit_test_full.log", "multi_result_test.log"]:
    shutil.copy(f"/content/{f}", f"/content/backup_bundle/{f}")
shutil.copy("/content/tt-mlir/build/bin/ttmlir-opt", "/content/backup_bundle/ttmlir-opt")
!tar -czf /content/backup_bundle/ttmlir-toolchain.tar.gz -C /opt ttmlir-toolchain
print("done:", os.listdir("/content/backup_bundle"))
